# HU-06 y HU-07 · Salas

[Inicio](../../tests.ipynb) · Anterior: [HU-05 · Analítica personal](hu05_analitica.ipynb) · Siguiente: [Integridad · Identidad y partidas](../integridad/identidad_y_partidas.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

**Administración (HU-06):** casos 16–22. **Ingreso y espera (HU-07):** casos 23–29.

## TC-FUNC-16

- **Resumen de la ficha:** Al crear una sala, esta queda en espera y la persona que la creó figura como anfitriona.
- **Método previsto:** API + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** La matriz partía de las pruebas de API y PostgreSQL (**U**). El 22 de noviembre de 2026, Chromium visible con PostgreSQL 16 desechable confirmó que la cuenta registrada no tenía sala (`GET /rooms/me`: 200, `null`); desde el formulario creó (`POST`: 201) la sala `MSBF9T` con partida rápida, inglés, dificultad media, 3 rondas y 90 s. La sala quedó en `waiting`, con `hostId: 1` y un único miembro, el creador, marcado `Host` y `You` en pantalla. El código coincidió con la URL y la consulta por código devolvió la misma sala (200); PostgreSQL tenía un índice único válido y no parcial sobre `rooms.code`.
- **Brecha o límite:** Una ejecución local, sin forzar una colisión ni probar ingreso de otro usuario. El lote de dos casos terminó con 1 aprobado, 1 fallido y código 1; este escenario aprobó.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Creación exitosa de una sala
    Dado que el usuario registrado no pertenece a otra sala
    Y selecciona el tipo de sala, idioma, dificultad, cantidad de rondas y tiempo límite
    Cuando solicita crear la sala
    Entonces el sistema crea una nueva sala en estado "esperando"
    Y asigna al usuario creador como anfitrión
    Y genera un código único para acceder a la sala
    Y agrega al anfitrión como miembro de la sala
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/room-administration.feature).

```gherkin
Feature: Administering multiplayer rooms

  @TC-FUNC-16
  Scenario: A registered player creates and hosts a waiting room
    Given I register a new account in the browser
    And I do not belong to a room
    When I select the room type, language, difficulty, rounds, and time limit and create the room
    Then the new room is waiting and lists me as its host and member
    And its unique access code opens that room
```

## TC-FUNC-17

- **Resumen de la ficha:** Se rechaza la creación de una sala si el usuario ya pertenece a una, sin cambiar su membresía.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Rechaza crear si ya pertenece a otra sala: `test_rooms_api.py` (**U**). [Fuente de prueba](../../backend/tests/test_rooms_api.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Intento de crear una sala perteneciendo a otra
    Dado que el usuario registrado ya pertenece a una sala
    Cuando intenta crear una nueva sala
    Entonces el sistema rechaza la creación de la sala
    Y mantiene al usuario en su sala actual
```

## TC-FUNC-18

- **Resumen de la ficha:** Los cambios válidos de configuración realizados por el anfitrión se reflejan en la sala en espera.
- **Método previsto:** API + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** El 22 de noviembre de 2026, TC-FUNC-18 pasó en Chromium visible con PostgreSQL 16 desechable (salida 0; 1 escenario y 6 pasos aprobados; 4,434 s; sin reintentos). En la sala en espera `QSKQHD`, el anfitrión cambió de español, dificultad fácil, 1 ronda y sin límite a inglés, dificultad media, 3 rondas y 90 s. Las cuatro solicitudes `PATCH` devolvieron 200. Tras recargar, una consulta autenticada explícita devolvió 200 y confirmó los valores; el lobby, que no se muestra hasta recibir el estado por WebSocket, volvió a mostrar al anfitrión y los cuatro controles con los valores nuevos. Se verificó que no quedaran contenedores, red ni listeners de la ejecución.
- **Historial:** En la primera ejecución local, que complementó la prueba de API (**U**), las cuatro solicitudes `PATCH` de la sala `5C3U64` devolvieron 200 y los controles reflejaron los cambios antes de recargar. El paso final agotó 5 s al esperar una consulta HTTP que el lobby no emite al recargar. La prueba ajustada sustituyó esa espera por una consulta autenticada explícita y comprobó el lobby tras recibir el estado. Traza de la primera ejecución: `frontend/e2e/test-results/scenario-ReB4z9/trace-1.zip`.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Modificación de la configuración por el anfitrión
    Dado que existe una sala en estado "esperando"
    Y el usuario es el anfitrión de la sala
    Cuando modifica el idioma, dificultad, cantidad de rondas o tiempo límite
    Entonces el sistema actualiza la configuración de la sala
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/room-administration.feature).

```gherkin
Feature: Administering multiplayer rooms

  @TC-FUNC-18
  Scenario: The host updates the rules of a waiting room
    Given I register a new account in the browser
    And I host a waiting room
    When I change the language, difficulty, rounds, and time limit through the lobby
    Then the updated rules persist and are visible after reloading the waiting lobby
```

## TC-FUNC-19

- **Resumen de la ficha:** Se rechaza el cambio de configuración solicitado por un miembro que no es anfitrión y la configuración queda intacta.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Un miembro no anfitrión no puede modificar la configuración: `test_rooms_api.py` (**U**). [Fuente de prueba](../../backend/tests/test_rooms_api.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Intento de modificación por un miembro
    Dado que existe una sala en estado "esperando"
    Y el usuario pertenece a la sala pero no es su anfitrión
    Cuando intenta modificar la configuración de la sala
    Entonces el sistema rechaza la modificación
    Y mantiene la configuración actual
```

## TC-FUNC-20

- **Resumen de la ficha:** Si el anfitrión expulsa a un jugador, este deja de pertenecer a la sala y se desconecta de ella.
- **Método previsto:** API + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** En el corte histórico (**P**), `test_rooms_api.py` y `test_ws_rooms.py` (**U**) cubrieron expulsión y desconexión. El 22 de noviembre, una corrida inicial observó la expulsión pero falló en el diagnóstico del segundo contexto. Tras corregir la inicialización de credenciales por origen, una corrida aislada con Chromium visible y PostgreSQL aprobó un escenario y ocho pasos sin reintentos; se verificó la limpieza. [Escenario](../../frontend/e2e/room-administration.feature).
- **Límite:** El recorrido cubre al anfitrión y al miembro expulsado, no una sala llena.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Expulsión de un jugador por el anfitrión
    Dado que existe una sala en estado "esperando"
    Y el usuario es el anfitrión de la sala
    Y existe otro jugador perteneciente a la sala
    Cuando el anfitrión expulsa al jugador
    Entonces el sistema elimina al jugador de los miembros de la sala
    Y el jugador deja de pertenecer a la sala
```

## TC-FUNC-21

- **Resumen de la ficha:** Si el anfitrión cierra la sala, se eliminan sus membresías y ya no se puede ingresar con ese código.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): [test_host_closes_the_room_and_everyone_stops_belonging](../../backend/tests/test_rooms_api.py) comprueba el 204 del cierre, la ausencia de membresía para ambos usuarios y el 404 al intentar usar el código después. El recorrido de navegador compartido con TC-USAB-02 complementa el cierre y la salida de ambos jugadores.
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Cierre de la sala por el anfitrión
    Dado que existe una sala administrada por el anfitrión
    Cuando el anfitrión decide cerrar la sala
    Entonces el sistema cierra la sala
    Y los jugadores dejan de pertenecer a ella
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/room-administration.feature).

```gherkin
Feature: Administering multiplayer rooms

  @TC-USAB-02
  Scenario: The host can cancel or confirm closing a waiting room
    Given I register a new account in the browser
    And I host a waiting room
    And another player belongs to the room in a separate browser context
    When I cancel closing the room with the Cancel button
    Then the player still belongs to the room
    When I confirm closing the room
    Then both players leave the closed room
```

El escenario de navegador no intenta reutilizar el código; el rechazo se verifica en la prueba de API. La salida voluntaria del anfitrión se documenta por separado en TC-FUNC-22.

## TC-FUNC-22

- **Resumen de la ficha:** Si el anfitrión abandona la sala, se eliminan la sala y sus membresías.
- **Método previsto:** API + Script WS.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Salida del anfitrión elimina la sala y desconecta a los participantes: `test_rooms_api.py`, `test_ws_rooms.py` (**U**). [Fuente de prueba](../../backend/tests/test_ws_rooms.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-06 #13](https://github.com/1Bnja/Typr/issues/13).

```gherkin
Característica: Administración de salas

  Escenario: Abandono de la sala por el anfitrión
    Dado que existe una sala activa
    Y el anfitrión pertenece a ella
    Cuando el anfitrión abandona la sala
    Entonces el sistema elimina la sala
    Y los demás jugadores dejan de pertenecer a ella
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-08
  Scenario: A member cannot close the room and the host can leave
    Given a room with two signed-in members
    When the member requests to close the room
    Then the request is forbidden and the waiting room is unchanged
    And PostgreSQL cascades a room deletion to its memberships
    When the host leaves the room
    Then the room and its memberships are gone
```

## TC-FUNC-23

- **Resumen de la ficha:** Con un código válido, el usuario ingresa a la sala en espera como miembro, no como anfitrión.
- **Método previsto:** API + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** En el corte histórico (**P**), `test_rooms_api.py` (**U**) cubrió ingreso y rol en API. El 22 de noviembre, Chromium visible y PostgreSQL aprobaron un escenario y ocho pasos sin reintentos: ingreso HTTP 200, sala en espera, rol de miembro, anfitrión visible y configuración de solo lectura. Se verificó la limpieza. [Escenario](../../frontend/e2e/room-administration.feature).
- **Límite:** Este recorrido no intenta directamente operaciones reservadas al anfitrión.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Ingreso exitoso mediante código
    Dado que el usuario no pertenece a otra sala
    Y existe una sala en estado "esperando" con menos de 10 jugadores
    Cuando el usuario ingresa el código válido de la sala
    Entonces el sistema agrega al usuario como miembro de la sala
    Y le asigna el perfil de miembro, sin permisos de anfitrión
    Y muestra al usuario los demás miembros y la configuración de la sala
```

## TC-FUNC-24

- **Resumen de la ficha:** Un código inválido no permite ingresar y se informa el error sin asignar una membresía.
- **Método previsto:** API + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** En el corte histórico (**P**), `test_rooms_api.py` (**U**) cubrió el rechazo por API. El 22 de noviembre, Chromium visible aprobó un escenario y siete pasos sin reintentos: HTTP 404, mensaje `That room code is not valid.`, permanencia en `/rooms` y ninguna membresía. Se verificó la limpieza. [Escenario](../../frontend/e2e/room-administration.feature).
- **Límite:** El escenario usa un código bien formado que no corresponde a ninguna sala.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Intento de ingreso con código inválido
    Dado que el usuario no pertenece a otra sala
    Cuando ingresa un código que no corresponde a una sala disponible
    Entonces el sistema rechaza el ingreso
    Y muestra un mensaje indicando que el código de sala no es válido
```

## TC-FUNC-25

- **Resumen de la ficha:** Una sala admite hasta 10 jugadores y rechaza al undécimo sin modificar la lista de miembros.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Admite diez jugadores y rechaza al undécimo en secuencia: `test_rooms_api.py` (**U**). Concurrencia se evalúa aparte en RI-07. [Fuente de prueba](../../backend/tests/test_rooms_api.py).
- **Brecha o límite:** La concurrencia se verifica por separado en RI-07.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Intento de ingreso a una sala llena
    Dado que existe una sala en estado "esperando" con 10 jugadores
    Cuando otro usuario intenta ingresar mediante su código
    Entonces el sistema rechaza el ingreso
    Y muestra un mensaje indicando que la sala está llena
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-07
  Scenario: Concurrent admissions compete for the last seat
    Given a waiting room with nine members
    When two new players request admission to the room concurrently
    Then one admission succeeds and the room has exactly ten members
```

El escenario relacionado prueba concurrencia por el último cupo; la API cubre el undécimo intento secuencial. Ninguna de esas pruebas comprueba el mensaje de sala llena en la interfaz.

## TC-FUNC-26

- **Resumen de la ficha:** Una sala con una partida en curso rechaza el ingreso de un nuevo miembro.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Rechaza ingreso con partida en curso: `test_rooms_api.py` (**U**). [Fuente de prueba](../../backend/tests/test_rooms_api.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Intento de ingreso a una sala con una partida en curso
    Dado que existe una sala en estado "en curso"
    Cuando un usuario intenta ingresar mediante su código
    Entonces el sistema rechaza temporalmente el ingreso
    Y el usuario debe esperar hasta que finalicen las rondas en curso
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-07
  Scenario: A new player cannot enter a live race
    Given a live race with two signed-in members
    Then a new player's admission is refused and the live roster is unchanged
```

## TC-FUNC-27

- **Resumen de la ficha:** Cuando un jugador confirma que está listo, el estado se refleja en la sala y se difunde a los miembros.
- **Método previsto:** Script WS + Manual.
- **Estado:** **V — Verificado**.
- **Evidencia:** En el corte histórico (**P**), `test_ws_rooms.py` (**U**) cubrió la difusión. El 22 de noviembre, dos contextos de Chromium visible y PostgreSQL aprobaron un escenario y ocho pasos sin reintentos: control presionado, `isReady=true` en la API y actualización en la lista del anfitrión sin recarga. Se verificó la limpieza. [Escenario](../../frontend/e2e/room-administration.feature).
- **Límite:** Se verificó con dos jugadores.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Confirmación de jugador listo
    Dado que el usuario pertenece a una sala en estado "esperando"
    Cuando presiona el botón "Listo"
    Entonces el sistema registra al jugador como listo
    Y muestra su estado actualizado a los demás miembros de la sala
```

## TC-FUNC-28

- **Resumen de la ficha:** Un usuario que ya pertenece a una sala no puede ingresar a otra y conserva su membresía original.
- **Método previsto:** API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): La API rechaza un segundo ingreso secuencial a otra sala: `test_rooms_api.py` (**U**). No prueba carrera entre solicitudes. [Fuente de prueba](../../backend/tests/test_rooms_api.py).
- **Brecha o límite:** La concurrencia se verifica por separado en RI-06.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Intento de pertenecer a dos salas simultáneamente
    Dado que el usuario ya pertenece a una sala
    Cuando intenta ingresar mediante el código de otra sala
    Entonces el sistema rechaza el ingreso
    Y mantiene al usuario en su sala actual
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-06
  Scenario: One player requests admission to two rooms concurrently
    Given two waiting rooms with different hosts
    When one new player requests admission to both rooms concurrently
    Then one admission succeeds and PostgreSQL stores only one membership for that player
```

El escenario relacionado prueba solicitudes concurrentes; no es la reproducción literal de un segundo ingreso secuencial.

## TC-FUNC-29

- **Resumen de la ficha:** Al finalizar la última ronda, la sala vuelve al estado de espera y se limpia el estado de listo de sus miembros.
- **Método previsto:** API + Script WS.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Tras la última ronda, WS notifica espera y reinicia los estados de listo; la API vuelve a admitir ingresos: `test_ws_rooms.py` (**U**). [Fuente de prueba](../../backend/tests/test_ws_rooms.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-07 #14](https://github.com/1Bnja/Typr/issues/14).

```gherkin
Característica: Ingreso y preparación en una sala

  Escenario: Retorno al estado de espera al finalizar las rondas
    Dado que una sala se encuentra en estado "en curso"
    Y se han completado todas las rondas configuradas
    Cuando finaliza la última ronda
    Entonces el sistema cambia el estado de la sala a "esperando"
    Y permite nuevamente el ingreso de jugadores
```

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · TC-FUNC-16 a TC-FUNC-29

**Plan (§5.5):** El anfitrión crea y administra su sala; un miembro no puede cambiar la configuración. El ingreso exige un código válido, cupo y una sala en espera, sin doble membresía. El estado de listo se comunica y la sala vuelve a espera tras la última ronda.

**Qué comprueba esta celda:** Ejecuta toda la suite de pruebas unitarias, integración y migraciones del backend. Para estas historias, `test_rooms_api.py` y `test_ws_rooms.py` incluyen estas aserciones:

- **16–19:** La creación deja al anfitrión en una sala en espera; otra creación con membresía activa devuelve 409. El anfitrión puede cambiar la configuración; el miembro recibe 403 y no la cambia.
- **20–22:** La expulsión retira y desconecta al miembro; el cierre retira las membresías; la salida del anfitrión elimina la sala y sus membresías.
- **23–26 y 28:** El código válido permite entrar como miembro; uno inexistente devuelve 404 y uno mal formado devuelve 422. El undécimo ingreso, el ingreso durante la carrera y el ingreso a una segunda sala devuelven 409.
- **27 y 29:** WebSocket difunde el estado de listo; tras la última ronda la sala vuelve a espera y se limpia ese estado.

**Límite:** Esta celda no ejecuta recorridos de navegador. Para seleccionar los recorridos de salas desde la raíz, prepare el [entorno de navegador](../../frontend/e2e/README.md) y ejecute `bash scripts/test-registration-browser --tags '@TC-FUNC-16 or @TC-FUNC-18 or @TC-FUNC-20 or @TC-FUNC-23 or @TC-FUNC-24 or @TC-FUNC-27' --retry 0 --format summary`. Use una sola etiqueta para reproducir un caso aislado.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-unit-integration-migrations')

[Inicio](../../tests.ipynb) · Anterior: [HU-05 · Analítica personal](hu05_analitica.ipynb) · Siguiente: [Integridad · Identidad y partidas](../integridad/identidad_y_partidas.ipynb)